# CUDA Aggregation Engine — Colab Backend

Runs the FastAPI backend and executes the original Numba CUDA kernels on the Google Colab GPU. The React dashboard connects through a **Cloudflare Quick Tunnel**.

**Run the cells from top to bottom after every Colab runtime restart.** Keep the Colab runtime running while using the dashboard.

| Step | Cells | Purpose |
|---|---:|---|
| 1 | 2–3 | Check the Tesla T4 GPU |
| 2 | 4–5 | Install the web-server packages |
| 3 | 6–7 | Upload `backend.zip` |
| 4 | 8–9 | Run the CUDA pre-flight tests |
| 5 | 10–11 | Start FastAPI and the Cloudflare tunnel |
| 6 | 12–13 | View server logs if needed |
| 7 | 14–15 | Stop the server and tunnel when finished |

**Cloudflare Quick Tunnel requires no account, token, or Colab Secrets.** The Backend URL and API key are newly generated whenever you start the backend. After a runtime restart, upload `backend.zip` again and update the dashboard's Connection page with the new URL and key.


## Step 1 · Check the GPU
If this does not show **Tesla T4**: *Runtime → Change runtime type → T4 GPU → Save*, then *Runtime → Restart session*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from numba import cuda
assert cuda.is_available(), "No GPU attached. Runtime > Change runtime type > T4 GPU, then restart the session."
print("Numba sees:", cuda.get_current_device().name)

## Step 2 · Install the web-server packages
Numba, NumPy and pandas are already in Colab — they are intentionally **not** reinstalled.

In [ ]:
!pip install -q fastapi uvicorn python-multipart pydantic psutil


## Step 3 · Upload the backend
Click **Choose files** and select `backend.zip` (it is in the project folder on your laptop). Re-running this cell replaces the old copy.

In [ ]:
import os, shutil, zipfile
from google.colab import files

shutil.rmtree("/content/backend", ignore_errors=True)
up = files.upload()
name = next(iter(up))
assert name.endswith(".zip"), "Please upload backend.zip"
with zipfile.ZipFile(name) as z:
    z.extractall("/content")
assert os.path.exists("/content/backend/app/kernels.py"), "backend.zip must contain a 'backend' folder"
print("Uploaded and extracted to /content/backend")
print(sorted(os.listdir("/content/backend")))

## Step 4 · Pre-flight: run your kernels on the GPU
Compiles the six kernels from `backend/app/kernels.py` (copied unchanged from your notebook) and checks them against a known answer (sum of 1…100,000 = 5,000,050,000).
All lines should say **True**.

In [ ]:
import sys
sys.path.insert(0, "/content/backend")
import colab_launcher as L

info = L.preflight()
print("GPU:", info["gpu"], "| is Tesla T4:", info["is_t4"], "| Numba", info["numba"])
for c in info["selftest"]:
    print(f'{c["op"]:>4}: expected {c["expected"]:>16,.0f}  GPU {c["gpu"]:>16,.0f}  pass={c["pass"]}')
print("Kernel self-test passed:", info["selftest_passed"])
assert info["selftest_passed"]

## Step 5 · Start FastAPI and the Cloudflare Quick Tunnel

Run the next code cell once the pre-flight tests pass. It will:

1. Generate a fresh API key.
2. Start the FastAPI backend.
3. Start a Cloudflare Quick Tunnel (no account or token required).
4. Check that the public URL can reach the backend.
5. Print the Backend URL and API key for the dashboard.

The Cloudflare URL and API key change when you restart the Colab runtime. Keep this runtime running while using the dashboard.


In [ ]:
import os
import sys
import time

# Confirm that the backend archive was uploaded and extracted.
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("Backend files are missing. Run the backend upload cell first.")

sys.path.insert(0, "/content/backend")
import colab_launcher as L

# Generate a fresh API key and allow the local React development server.
API_KEY = L.new_api_key()
ALLOWED_ORIGINS = "http://localhost:5173,http://127.0.0.1:5173"

# Clean up any tunnel from an earlier run in this same runtime.
L.stop_tunnels()

# Start FastAPI and then create the Cloudflare Quick Tunnel.
L.start_server(API_KEY, ALLOWED_ORIGINS)
PUBLIC_URL = L.start_cloudflared()

print("Cloudflare tunnel URL:", PUBLIC_URL)
print("Checking public connection (DNS may take a few seconds to become ready)...")

# Retry briefly in case the new Cloudflare hostname has not propagated yet.
check = None
last_error = None
for attempt in range(1, 7):
    try:
        check = L.check_public(PUBLIC_URL, API_KEY)
        break
    except Exception as exc:
        last_error = exc
        print(f"Connection attempt {attempt}/6 did not succeed: {exc}")
        if attempt < 6:
            time.sleep(5)

if check is None:
    raise RuntimeError(
        "The backend and Cloudflare tunnel started, but the public connection check failed. "
        f"Keep this runtime running and inspect the Cloudflare/server logs. Last error: {last_error}"
    )

print("\nBackend successfully connected!")
print("GPU:", check.get("name", "Unknown"))
print("Warm-up:", check.get("warmup", "Completed"))
print("=" * 64)
print("Backend URL:", PUBLIC_URL)
print("API KEY:", API_KEY)
print("=" * 64)
print("Paste both into your dashboard: Connection page > Save and test connection.")


## Server log (run any time)
Useful if something fails. Each request from the dashboard appears here.

In [ ]:
import os, sys
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("The backend files are not in this Colab session (a restart erases them). Run Step 3 (upload backend.zip) first.")
sys.path.insert(0, "/content/backend")
import colab_launcher as L

print(L.tail_log(40))

## Stop everything
Run this when you are finished (also frees the GPU).

In [ ]:
import os, sys
if not os.path.exists("/content/backend/colab_launcher.py"):
    raise RuntimeError("The backend files are not in this Colab session (a restart erases them). Run Step 3 (upload backend.zip) first.")
sys.path.insert(0, "/content/backend")
import colab_launcher as L

L.stop_tunnels(); L.stop_server(); print('Stopped.')